# Xenopus embryo example (legacy)

> **Prefer the maintained tutorials:**
> - [`01_quickstart_synthetic.ipynb`](01_quickstart_synthetic.ipynb) — runs without external files
> - [`02_PBMC3k_full_workflow.ipynb`](02_PBMC3k_full_workflow.ipynb) — full API with PBMC3k demo data

This notebook uses pickled matrices referenced below; place them next to the notebook or adjust paths.


In [ ]:
from scevonet import net
import seaborn as sns
import pandas as pd
import networkx as nx
import matplotlib as mpl
import matplotlib.pyplot as plt

# Read data: count matrices and cluster lists

You can download test_data [here](https://www.dropbox.com/sh/tt1cgrxp2qu9r0f/AADJhRbiWTwebLhD8dPfAJLCa?dl=0)

In [ ]:
gastrulation_df = pd.read_pickle('test_data_12.pkl')
neurulation_df = pd.read_pickle('test_data_20.pkl')

In [ ]:
gastrulation_df.head(3)

# Generate the cell type models

Two arguments: count matrix and cluster list. For each cluster scEvoNet generates model and saves top important features (genes)

In [ ]:
gastrulation_obj = net.Sample(
    matrix=gastrulation_df[[x for x in gastrulation_df if x != 'clusters']],
    cell_types=gastrulation_df['clusters'])

In [ ]:
neurulation_obj = net.Sample(
    matrix=neurulation_df[[x for x in neurulation_df if x != 'clusters']],
    cell_types=neurulation_df['clusters'])

# Predict cell types similarity with generated models

In [ ]:
gastrulation_vs_neurulation = net.EvoManager(gastrulation_obj, neurulation_obj)

## Get comparison df

In [ ]:
comparison_df = gastrulation_vs_neurulation.generate_comparison_df()

__Get clustermap for df.T as we want to see correlations of predctions results of the multiple models__


In [ ]:
sns.clustermap(comparison_df.T.corr(), yticklabels=True, figsize=(10, 10),cmap='viridis')

To study what genes connect cell types and that are the possible co-opted genes you can generate subnetwork from the short paths from one cell type to another cell type

__closets_clusters__ argument controls how many similar cell types (from the __comparison_df__) are involved into generation of the subnetwork

argument __number_of_short_paths__ controls the size of the subnetwork

In [ ]:
subnetwork = gastrulation_vs_neurulation.generate_cell_type_network(
                              '20_neural crest',
                              '12_neural crest',
                              number_of_shortest_paths=50,closest_clusters=20)

Now you can plot the subnetwork

In [ ]:
mpl.rcParams['figure.dpi'] = 100
plt.figure(figsize=(10, 10))
net.draw_net(subnetwork, subnetwork.nodes)

You can also get subnetwork in the form of the dataframe with argument *net=False*

In [ ]:
subnetwork = gastrulation_vs_neurulation.generate_cell_type_network(
                              '20_neural crest',
                              '12_neural crest',
                              number_of_shortest_paths=50,closest_clusters=20, net=False)
subnetwork.head()

To get only direct connections (with one gene between two cell types) you can specify __get_only_direct__ argument


In [ ]:
subnetwork = gastrulation_vs_neurulation.generate_cell_type_network(
                              '20_neural crest',
                              '12_neural crest',
                              number_of_shortest_paths=50,
                              closest_clusters=20, get_only_direct=True)

In [ ]:
plt.figure(figsize=(10, 10))
net.draw_net(subnetwork, subnetwork.nodes)